# Week 3 Homework 

## Purpose of Homework

This homework assignment will help you practice building and evaluating a linear regression model for prediction. You will predict movie popularity using features that are different from those used in lecture and lab.

You are encouraged to refer to lecture content and liberally use course resources such as the discussion board and office hours.

## Logistics

Due date: The homework is due 12:00pm on Thursday, January 29, 2026.

You will submit your homework on [MarkUs](https://markus.teach.cs.toronto.edu/markus/). 

1. Download this file (`STA272_hw3_student.ipynb`) from JupyterHub. (See [our JupyterHub Guide](../guides/jupyterhub_guide.ipynb) for detailed instructions.)
2. Submit this file to MarkUs under the hw3 assignment. (See [our MarkUs Guide](../guides/markus_guide.ipynb) for detailed instructions.)

All homeworks will take place in a Jupyter notebook (like this one). When you are done, you will download this notebook and submit it to MarkUs.

## Predicting Movie Popularity

In lecture and lab, we predicted movie revenue using features like budget, popularity, and genre indicators. In this homework, you will flip the problem around and **predict popularity** using different features.

Complete the following tasks and answer the questions.

## Task #1

Read `movies_modeling.csv` into a pandas dataframe called `movies_df`.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import re

# Read the CSV file into a dataframe
movies_df = pd.read_csv('...')

# Display the first few rows
movies_df.head()

## Task #1b

Make a histogram of the variable `popularity`

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
movies_df["..."].hist(bins=50)
plt.xlabel("...")
plt.ylabel("Count")
plt.show()

## Task #2

Create a binary indicator variable called `popular_genres` that equals 1 if a movie has at least one of the following genres: `Action`, `Adventure`, `Comedy`, or `Drama`, and 0 otherwise.

In [ ]:
# Create popular_genres indicator
popular_genre_list = [...]
movies_df['popular_genres'] = movies_df['genres'].fillna('').apply(
    lambda x: int(any(... in x for ... in ...))
)

# Check your work
movies_df[['title', 'genres', 'popular_genres']].head(10)

## Task #3

The `release_date` column contains dates in the format `YYYY-MM-DD` (e.g., `2023-12-16`).

Using `release_date` create the following features:

1. `release_month` - extract the month (1-12) from the `release_date` column
2. `release_year` - extract the year from the `release_date` column
3. `summer_release` - a binary indicator (1 or 0) for whether the movie was released in summer (June, July, or August, i.e., months 6, 7, or 8)


**Hints:** 
- Use `pd.to_datetime()` to convert the date string, then use `.dt.month` and `.dt.year` to extract components

In [ ]:
# Convert release_date to datetime and extract month
movies_df['release_month'] = pd.to_datetime(movies_df['...']).dt...

# Create release_year 
movies_df['release_year'] = pd.to_datetime(movies_df['...']).dt...

# Create summer_release indicator (months 6, 7, 8)
movies_df['summer_release'] = movies_df['release_month'].isin([..., ..., ...]).astype(...)

# Check your work
movies_df[['title', 'release_date', 'release_month', 'release_year', 'summer_release']].head(10)

## Task #4

The `actors` column lists actor names followed by popularity scores in parentheses (e.g., `Sam Worthington(6.2), Zoe Saldaña(9.9)`). 

Use `re.findall()` with the pattern `r'\((\d+\.?\d*)\)'` to extract popularity scores from each row. Then create a new variable, `actor_max_pop`, equal to the **maximum** popularity score found in that row's `actors` string.

In [ ]:
def extract_max_popularity(actors_string):
    """Extract the maximum popularity score from the actors string."""
    if pd.isna(actors_string):
        return np.nan
    # Use regex to find all numbers in parentheses
    scores = re.findall(r'\((\d+\.?\d*)\)', actors_string)
    if scores:
        return max([float(s) for s in scores])
    return np.nan

# Apply the function to create actor_max_pop
movies_df['actor_max_pop'] = movies_df['...'].apply(...)

# Check your work
movies_df[['title', 'actors', 'actor_max_pop']].head()

## Task #5

For our model, we will use `popularity` as the response (y) and the following as predictors (X):
- `revenue`
- `vote_average`
- `runtime`
- `release_month`
- `release_year`
- `summer_release`
- `actor_max_pop`
- `cast_count`
- `crew_count`
- `popular_genres`

Check the number of missing values in these key variables. Then remove any rows with missing values in these variables and store the result in `movies_df_clean`.

In [ ]:
# Define key variables
key_vars = ['popularity', 'revenue', 'cast_count', 'crew_count', 'release_month', 'release_year',
            'vote_average', 'runtime', 'summer_release', 'actor_max_pop', 'popular_genres']

# Check missing values
print("Missing values in key variables:")
print(movies_df[key_vars].isnull()...())

# Remove rows with missing values
movies_df_clean = movies_df.dropna(subset=...)

print(f"\nOriginal dataset size: {len(movies_df)}")
print(f"Clean dataset size: {len(movies_df_clean)}")

## Task #6

Using `sklearn.model_selection.KFold`, set up **10-fold cross validation**. Set your **seed** by specifying `random_state = 123` and use `shuffle = True`.

In [ ]:
from sklearn.model_selection import KFold

# Define X and y
X = movies_df_clean[['revenue', 'vote_average', 'runtime', 'release_month', 'release_year', 
                      'summer_release', 'actor_max_pop', 'cast_count', 'crew_count', 'popular_genres']]
y = movies_df_clean['...']

# Set up 10-fold cross validation
kf = KFold(n_splits=..., shuffle=..., random_state=...)

print(f"Number of folds: {kf.get_n_splits()}")
print(f"Total samples: {len(X)}")

## Task #7

Fit a linear regression model using `statsmodels` and perform 10-fold cross validation. For each fold, fit the model on the training data and calculate the $R^2$ on the test data. 
Report the $R^2$ and RMSE for each fold and the mean cross-validated $R^2$ and RMSE.

**Hint:** Remember to add a constant term using `sm.add_constant()`.

In [ ]:
import statsmodels.api as sm
from sklearn.metrics import r2_score, root_mean_squared_error

# Store results for each fold
cv_r2_scores = []
cv_rmse_scores = []
y_pred_all = np.full(len(y), np.nan)  # Store predictions for all observations

# Perform 10-fold cross validation
for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    # Split data
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[...], y.iloc[...]
    
    # Add constant
    X_train_const = sm.add_constant(...)
    X_test_const = sm.add_constant(...)
    
    # Fit model
    model = sm.OLS(..., ...).fit()
    
    # Predict on test set
    y_pred = model.predict(...)
    
    # Store predictions
    y_pred_all[test_idx] = y_pred
    
    # Calculate metrics
    r2 = r2_score(..., ...)
    rmse = root_mean_squared_error(..., ...)
    
    cv_r2_scores.append(r2)
    cv_rmse_scores.append(rmse)
    
    print(f"Fold {fold}: R² = {r2:.4f}, RMSE = {rmse:.4f}")

print(f"\nMean CV R²: {np.mean(cv_r2_scores):.4f}")

## Task #8

Using the stored predictions from cross-validation, create a scatter plot of actual vs predicted popularity values.

Label the x-axis as "Actual Popularity", the y-axis as "Predicted Popularity", and the title as "Actual vs Predicted Popularity (10-Fold CV)".

**Store the resulting matplotlib figure in a variable called `pred_plot_fig`.**

In [ ]:
import matplotlib.pyplot as plt

# Create the scatter plot
plt.figure(figsize=(8, 8))

# Plot actual vs predicted
plt.scatter(..., ..., alpha=0.5)

# Add 45-degree reference line
min_val = min(y.min(), y_pred_all.min())
max_val = max(y.max(), y_pred_all.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', label='Perfect Prediction')

# Add labels and title
plt.xlabel('...')
plt.ylabel('...')
plt.title('...')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Store the figure
pred_plot_fig = plt.gcf()

## Question #1

Based on the cross-validated $R^2$ and RMSE from Tasks #6 and #7, how well does the linear regression model predict movie popularity? What do these metrics tell you about the model's performance?

**To obtain full marks (4 points):**
- Correctly state the mean cross-validated $R^2$ and RMSE values (1 point)
- Explain what $R^2$ measures in the context of prediction (1 point)
- Explain what RMSE measures and why it's useful (1 point)
- Discuss whether these values indicate good or poor predictive performance (1 point)

### Student Answer:

[Write your answer here]

## Question #2

Based on the scatter plot from Task #8, describe how well the model predicts popularity. Are there any patterns in the prediction errors (e.g. is there over-prediction or under-prediction for certain ranges of popularity)?

**To obtain full marks (4 points):**
- Describe the overall pattern in the scatter plot (1 point)
- Identify any systematic patterns in the prediction errors (1 point)
- Explain why 10-fold cross validation gives a more reliable estimate of model performance than a single train-test split (1 point)
- Suggest at least one way the model could potentially be improved (1 point)


### Student Answer:

[Write your answer here]